In [1]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Initialize the Kaggle secrets client
user_secrets = UserSecretsClient()

# 2. Retrieve your HF_TOKEN secret securely
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    
    # 3. Set it as an environment variable for Hugging Face libraries
    os.environ["HF_TOKEN"] = hf_token
    
    # 4. Authenticate the session
    login(token=hf_token)
    print("Successfully authenticated with Hugging Face!")
    
except Exception as e:
    print(f"Error loading HF_TOKEN from Kaggle Secrets: {e}")
    print("Make sure you added the secret with the exact label 'HF_TOKEN' under Add-ons > Secrets.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated with Hugging Face!


In [2]:
!git clone -b cot https://github.com/bharathgaddam1712/cot-compliance-safety.git
%cd /kaggle/working/cot-compliance-safety

Cloning into 'cot-compliance-safety'...
remote: Enumerating objects: 169, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 169 (delta 95), reused 157 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (169/169), 10.80 MiB | 21.19 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/kaggle/working/cot-compliance-safety


In [3]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.1/619.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.0/233.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.0/273.0 kB 20.4 MB/s eta 0:00:00


In [4]:
MODELS = [
    "deepseek-r1-7b",
    "deepseek-r1-8b",
    "deepseek-r1-14b",
    "qwen3-14b",
    "qwen3-4b",
    "qwen3-8b",
    "deepseek-r1-32b",
    "qwq-32b"
]

BATCH = "without_attacks"   # this notebook is scoped to this one batch

# judge-model generation batch size (renamed from --batch in the scripts
# themselves to avoid clashing with the with_attacks/without_attacks flag)
GEN_BATCH = 8

# RTCP: max Q3 rows per model to analyze (None = all). Start with a small
# number on the first run to sanity-check timing before doing a full pass.
RTCP_LIMIT = None

In [5]:
import glob
import shutil
from pathlib import Path

GEN_DIR = Path("/kaggle/working/cot-compliance-safety/data/without_attacks/generations")
GEN_DIR.mkdir(parents=True, exist_ok=True)

patterns = [
    "/kaggle/input/*/data/without_attacks/generations/generations_*.jsonl",
    "/kaggle/input/*/data/raw/generations_*.jsonl",   # old flat layout, pre-restructure
    "/kaggle/input/*/generations_*.jsonl",
    "/kaggle/input/*/*.jsonl",
]
copied = []
for pattern in patterns:
    for src in glob.glob(pattern):
        name = Path(src).name
        if not name.startswith("generations_"):
            continue
        dst = GEN_DIR / name
        if not dst.exists() or dst.stat().st_size < Path(src).stat().st_size:
            shutil.copy2(src, dst)
            copied.append(name)

print("GEN_DIR:", GEN_DIR)
print("generation files:", sorted(p.name for p in GEN_DIR.glob("generations_*.jsonl")))
print("newly copied:", sorted(set(copied)))

GEN_DIR: /kaggle/working/cot-compliance-safety/data/without_attacks/generations
generation files: ['generations_deepseek-r1-14b.jsonl', 'generations_deepseek-r1-32b.jsonl', 'generations_deepseek-r1-7b.jsonl', 'generations_deepseek-r1-8b.jsonl', 'generations_qwen3-14b.jsonl', 'generations_qwen3-4b.jsonl', 'generations_qwen3-8b.jsonl', 'generations_qwq-32b.jsonl']
newly copied: []


In [6]:
%cd /kaggle/working/cot-compliance-safety/src
!python -c "from config import BASE, JOB_LIST, batch_dirs; d = batch_dirs('without_attacks'); print('BASE=', BASE); print('JOB_LIST=', JOB_LIST); [print(k, '=', v) for k, v in d.items()]"

/kaggle/working/cot-compliance-safety/src
BASE= /kaggle/working/cot-compliance-safety
JOB_LIST= /kaggle/working/cot-compliance-safety/data/job_list.jsonl
generations = /kaggle/working/cot-compliance-safety/data/without_attacks/generations
cot = /kaggle/working/cot-compliance-safety/data/without_attacks/cot
final = /kaggle/working/cot-compliance-safety/data/without_attacks/final
labeled = /kaggle/working/cot-compliance-safety/data/without_attacks/labeled


In [7]:
models_arg = " ".join(MODELS)
!python label_pipeline.py --models {models_arg} --batch {BATCH}

Loading judge: Qwen/Qwen2.5-7B-Instruct
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.23MB/s]
tokenizer_config.json: 7.30kB [00:00, 2.72MB/s]
vocab.json: 2.78MB [00:00, 59.9MB/s]
merges.txt: 1.67MB [00:00, 147MB/s]
tokenizer.json: 7.03MB [00:00, 132MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 27.8kB [00:00, 78.2MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:04<00:00, 16.12s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:04<00:00, 449MB/s]
Loading weights:   0%|                                  | 0/339 [00:00<?, ?it/s]
Loading weights:   0%| | 1/339 [00:00<00:00, 2851.33it/s, Materializing param=lm
Loading weights:   0%| | 1/339 [00:00<00:00, 1831.57it/s, Materializing param=lm
Loading weights:   1%| | 2/339 [00:04<13:34,  2.42s/it, Materializing param=lm_h
Loading weights:   1%| | 2/339 [00:04<13:34,  2.42s/it, Materializing param=mode
Loading weights:   1%| | 2/339 [00:04<13:34,  

In [8]:
limit_arg = f"--limit {RTCP_LIMIT}" if RTCP_LIMIT is not None else ""
for model in MODELS:
    !python rtcp.py --model {model} --batch {BATCH} {limit_arg}

Loading judge: Qwen/Qwen2.5-7B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 339/339 [00:09<00:00, 37.13it/s, Materializing param=mo
[judge] loaded Qwen/Qwen2.5-7B-Instruct
[deepseek-r1-7b] resuming — 0 already done
[deepseek-r1-7b] Q3 rows to process: 66
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  [0/66] ea8d199d05ed  rtcp=0.00%  checked=1
  [5/66] f2373089ec4d  rtcp=73.90%  checked=6
  [10/66] 3b42857353e6  rtcp=None  checked=6
  [15/66] 07dfd1aa8b69  rtcp=0.00%  checked=1
  [20/66] 97bb72258830  rtcp=None  checked=8
  [25/66] 694c692fa281  rtcp=0.00%  checked=1
  [30/66] 7b33859134cf  rtcp=0.00%  checked=1
  [35/66] 1c37f9312e4a  rtcp=0.00%  checked=1
  [40/66] c8fd073e8797  rtcp=0.00%  checked=1
  [45/66] abab50f9be7d  rtcp=0.00%  checked=1
  [50/66] acdcfdba3b0a  rtcp=0.00%  checked=1
  [55/66] 000ae04dece3  rtcp=0.00%  checked=1
  [60/66] ea50adf081

In [9]:
from pathlib import Path
import json
import statistics
from collections import Counter

LABELED_DIR = Path(f"/kaggle/working/cot-compliance-safety/data/{BATCH}/labeled")
RTCP_DIR    = Path(f"/kaggle/working/cot-compliance-safety/data/rtcp/{BATCH}")

print(f"=== quadrants ({BATCH}) ===")
for model in MODELS:
    path = LABELED_DIR / f"labeled_{model}.jsonl"
    if not path.exists():
        print(f"[missing] {path.name}")
        continue
    quads = Counter()
    with open(path) as f:
        for line in f:
            if line.strip():
                quads[json.loads(line).get("compliance_quadrant")] += 1
    print(f"{path.name}: {sum(quads.values())} rows  quadrants={dict(quads)}")

print(f"\n=== RTCP ({BATCH}) ===")
for model in MODELS:
    path = RTCP_DIR / f"rtcp_{model}.jsonl"
    if not path.exists():
        print(f"[missing] {path.name}")
        continue
    pcts = []
    n_total = 0
    with open(path) as f:
        for line in f:
            if line.strip():
                n_total += 1
                r = json.loads(line)
                if r.get("rtcp_pct") is not None:
                    pcts.append(r["rtcp_pct"])
    mean_str = f"{statistics.mean(pcts):.2%}" if pcts else "n/a"
    print(f"{path.name}: {n_total} Q3 rows analyzed, {len(pcts)} commitment points found, mean RTCP%={mean_str}")

=== quadrants (without_attacks) ===
labeled_deepseek-r1-7b.jsonl: 1013 rows  quadrants={'Q1': 500, 'Q4': 290, 'Q2': 157, 'Q3': 66}
labeled_deepseek-r1-8b.jsonl: 1013 rows  quadrants={'Q1': 595, 'Q4': 191, 'Q2': 151, 'Q3': 76}
labeled_deepseek-r1-14b.jsonl: 1013 rows  quadrants={'Q4': 217, 'Q1': 575, 'Q3': 143, 'Q2': 78}
labeled_qwen3-14b.jsonl: 1013 rows  quadrants={'Q1': 876, 'Q2': 118, 'Q3': 14, 'Q4': 5}
labeled_qwen3-4b.jsonl: 1013 rows  quadrants={'Q1': 836, 'Q2': 154, 'Q4': 15, 'Q3': 8}
labeled_qwen3-8b.jsonl: 1013 rows  quadrants={'Q1': 840, 'Q2': 155, 'Q4': 9, 'Q3': 9}
labeled_deepseek-r1-32b.jsonl: 1013 rows  quadrants={'Q1': 639, 'Q4': 185, 'Q2': 108, 'Q3': 81}
labeled_qwq-32b.jsonl: 1013 rows  quadrants={'Q1': 829, 'Q2': 51, 'Q3': 79, 'Q4': 54}

=== RTCP (without_attacks) ===
rtcp_deepseek-r1-7b.jsonl: 66 Q3 rows analyzed, 63 commitment points found, mean RTCP%=1.40%
rtcp_deepseek-r1-8b.jsonl: 76 Q3 rows analyzed, 74 commitment points found, mean RTCP%=3.12%
rtcp_deepseek-r1-